# Random kmin Dataset Generation (Local PC)

This notebook generates a dataset with **random `k_min` per image** using your local Python environment.

- Total images: `32000`
- Resolution: `128x128`
- `k_min`: uniform random integer in `[1, 32]`
- `k_max`: auto (Nyquist)

In [1]:
from pathlib import Path
import os
import sys
import subprocess

project_root = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
script_path = project_root / 'scripts' / 'run_generation_random_kmin.py'
output_dir = project_root / 'data' / 'raw'

print('Project root:', project_root)
print('Generator script exists:', script_path.exists())
print('Python executable:', sys.executable)

Project root: /home/davas/Documents/cnn_astro
Generator script exists: True
Python executable: /home/davas/.uve/py39/bin/python


In [2]:
# If this fails, install missing packages in this notebook kernel:
# %pip install numpy h5py pyFC

import numpy as np
import h5py
print('numpy version:', np.__version__)
print('h5py version:', h5py.__version__)

numpy version: 2.0.2
h5py version: 3.14.0


In [3]:
# Generation settings (edit if needed)
NUM_IMAGES = 32000
IMAGE_WIDTH = 128
IMAGE_HEIGHT = 128
KMIN_LOW = 1
KMIN_HIGH = 32
BATCH_SIZE = 256
NUM_WORKERS = max(1, (os.cpu_count() or 4) - 2)

env = os.environ.copy()
env.update({
    'NUM_IMAGES': str(NUM_IMAGES),
    'IMAGE_WIDTH': str(IMAGE_WIDTH),
    'IMAGE_HEIGHT': str(IMAGE_HEIGHT),
    'KMIN_LOW': str(KMIN_LOW),
    'KMIN_HIGH': str(KMIN_HIGH),
    'BATCH_SIZE': str(BATCH_SIZE),
    'NUM_WORKERS': str(NUM_WORKERS),
})

if not script_path.exists():
    raise FileNotFoundError(f'Missing script: {script_path}')

cmd = [sys.executable, str(script_path)]
print('Running command:', ' '.join(cmd))
print('Workers:', NUM_WORKERS)
print('This may take a while for 32k images...')

result = subprocess.run(cmd, cwd=str(project_root), env=env, check=False)
print('Exit code:', result.returncode)
if result.returncode != 0:
    raise RuntimeError('Dataset generation failed. Check notebook output above for details.')

Running command: /home/davas/.uve/py39/bin/python /home/davas/Documents/cnn_astro/run_generation_random_kmin.py
Workers: 14
This may take a while for 32k images...
Random-k_min dataset generation
Output: /home/davas/Documents/cnn_astro/generated_datasets/randomkmin_1-32_128x128_32000.h5
Images: 32000
Resolution: 128x128
k_min range: [1, 32] (uniform continuous)
Workers: 14


/home/davas/.uve/py39/lib/python3.9/site-packages/pyFC/clouds.py:1229: RuntimeWarning: divide by zero encountered in power
  return np.where(np.greater_equal(np.abs(k), kmin), np.abs(k) ** (beta - 2.), 0)
/home/davas/.uve/py39/lib/python3.9/site-packages/pyFC/clouds.py:1229: RuntimeWarning: divide by zero encountered in power
  return np.where(np.greater_equal(np.abs(k), kmin), np.abs(k) ** (beta - 2.), 0)
/home/davas/.uve/py39/lib/python3.9/site-packages/pyFC/clouds.py:1229: RuntimeWarning: divide by zero encountered in power
  return np.where(np.greater_equal(np.abs(k), kmin), np.abs(k) ** (beta - 2.), 0)
/home/davas/.uve/py39/lib/python3.9/site-packages/pyFC/clouds.py:1229: RuntimeWarning: divide by zero encountered in power
  return np.where(np.greater_equal(np.abs(k), kmin), np.abs(k) ** (beta - 2.), 0)
/home/davas/.uve/py39/lib/python3.9/site-packages/pyFC/clouds.py:1229: RuntimeWarning: divide by zero encountered in power
  return np.where(np.greater_equal(np.abs(k), kmin), np.a

Batch 1/125 | 256/32000 (0.8%) | 152.39 img/s | ETA 3.5 min
Batch 2/125 | 512/32000 (1.6%) | 163.47 img/s | ETA 3.2 min
Batch 3/125 | 768/32000 (2.4%) | 167.11 img/s | ETA 3.1 min
Batch 4/125 | 1024/32000 (3.2%) | 169.63 img/s | ETA 3.0 min
Batch 5/125 | 1280/32000 (4.0%) | 171.50 img/s | ETA 3.0 min
Batch 6/125 | 1536/32000 (4.8%) | 172.59 img/s | ETA 2.9 min
Batch 7/125 | 1792/32000 (5.6%) | 173.06 img/s | ETA 2.9 min
Batch 8/125 | 2048/32000 (6.4%) | 173.91 img/s | ETA 2.9 min
Batch 9/125 | 2304/32000 (7.2%) | 174.29 img/s | ETA 2.8 min
Batch 10/125 | 2560/32000 (8.0%) | 174.64 img/s | ETA 2.8 min
Batch 11/125 | 2816/32000 (8.8%) | 174.48 img/s | ETA 2.8 min
Batch 12/125 | 3072/32000 (9.6%) | 174.49 img/s | ETA 2.8 min
Batch 13/125 | 3328/32000 (10.4%) | 174.52 img/s | ETA 2.7 min
Batch 14/125 | 3584/32000 (11.2%) | 174.28 img/s | ETA 2.7 min
Batch 15/125 | 3840/32000 (12.0%) | 174.88 img/s | ETA 2.7 min
Batch 16/125 | 4096/32000 (12.8%) | 175.10 img/s | ETA 2.7 min
Batch 17/125 | 4

In [4]:
# Verify output file and k_min distribution
import numpy as np
import h5py

expected_name = f'randomkmin_{KMIN_LOW}-{KMIN_HIGH}_{IMAGE_HEIGHT}x{IMAGE_WIDTH}_{NUM_IMAGES}.h5'
dataset_file = output_dir / expected_name

if not dataset_file.exists():
    raise FileNotFoundError(f'Expected dataset not found: {dataset_file}')

with h5py.File(dataset_file, 'r') as f:
    images_shape = f['images'].shape
    kmins = f['parameters']['k_min'][:]

print('Dataset file:', dataset_file)
print('Images shape:', images_shape)
print('k_min min/max:', float(kmins.min()), float(kmins.max()))
print('k_min mean/std:', float(kmins.mean()), float(kmins.std()))

hist_counts, hist_edges = np.histogram(kmins, bins=10, range=(KMIN_LOW, KMIN_HIGH))
print('k_min histogram (10 bins):')
for i, c in enumerate(hist_counts):
    left = hist_edges[i]
    right = hist_edges[i + 1]
    print(f'  [{left:5.2f}, {right:5.2f}): {int(c)}')

Dataset file: /home/davas/Documents/cnn_astro/generated_datasets/randomkmin_1-32_128x128_32000.h5
Images shape: (32000, 128, 128)
k_min min/max: 1.0009585618972778 31.999433517456055
k_min mean/std: 16.45889663696289 8.924945831298828
k_min histogram (10 bins):
  [ 1.00,  4.10): 3227
  [ 4.10,  7.20): 3201
  [ 7.20, 10.30): 3190
  [10.30, 13.40): 3229
  [13.40, 16.50): 3166
  [16.50, 19.60): 3233
  [19.60, 22.70): 3264
  [22.70, 25.80): 3148
  [25.80, 28.90): 3214
  [28.90, 32.00): 3128
